# RoadBuddy Phase 2 — Notebook 01
## Canonical Multi-frame MVP for V100

This notebook is the canonical training/evaluation template used by Notebook 02.

Core guarantees:

- V100-safe precision: FP16 on, BF16 off.
- Dynamic RAM and VRAM guards.
- Multi-frame preprocessing with local tile caps.
- Answer-preserving label masking.
- Trainer API compatibility across Transformers versions.
- `num_patches_list` never forwarded to `model.forward()`.
- Stable output contract for Notebook 02 and Notebook 03.


## Canonical configuration

In [1]:
# =========================
# CANONICAL PHASE 2 CONFIG
# =========================

RUN_ID = 'f1_uniform_r16_lr1e4_debug'
STAGE = 'frame_count'
TARGET_SET = 'attention'

DEBUG_MODE = True
DEBUG_TRAIN_SAMPLES = 256
DEBUG_VAL_SAMPLES = 64

NUM_FRAMES = 1
SAMPLING_STRATEGY = 'uniform'
MAX_TILES_PER_FRAME = 5
MAX_TOTAL_TILES = NUM_FRAMES * MAX_TILES_PER_FRAME

INPUT_SIZE = 448
MAX_LENGTH = 2048
MAX_NEW_TOKENS = 4

LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 0.0001
NUM_TRAIN_EPOCHS = 1

VERBOSE_PREPROCESSING = False
OUTPUT_FORMAT_INSTRUCTION = (
    "Chỉ trả lời duy nhất một ký tự A, B, C hoặc D. "
    "Không giải thích và không thêm nội dung khác."
)


## 1. Imports và reproducibility

In [2]:
import gc
import inspect
import json
import os
import random
import re
import time
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import transformers
from torch.utils.data import Dataset, Subset
from transformers import AutoModel, Trainer, TrainingArguments, PreTrainedModel
from peft import LoraConfig, get_peft_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required.")

device = torch.device("cuda:0")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


PyTorch: 2.5.1+cu118
CUDA: 11.8
GPU: Tesla V100-SXM2-32GB
VRAM GB: 31.73


### Runtime metadata

In [3]:
import sys
from pathlib import Path

def build_runtime_metadata():
    return {
        "python_version": sys.version,
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "gpu_name": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
    }

RUNTIME_METADATA = build_runtime_metadata()
print(json.dumps(RUNTIME_METADATA, ensure_ascii=False, indent=2))


{
  "python_version": "3.12.13 (main, Jul 29 2026, 16:29:48) [GCC 11.4.0]",
  "torch_version": "2.5.1+cu118",
  "transformers_version": "5.14.1",
  "gpu_name": "Tesla V100-SXM2-32GB",
  "cuda_available": true,
  "cuda_version": "11.8"
}


## GPU memory guard

In [4]:
import gc
import time

def cuda_memory_snapshot(device_index=0):
    if not torch.cuda.is_available():
        return {"cuda_available": False}
    torch.cuda.synchronize(device_index)
    free_b, total_b = torch.cuda.mem_get_info(device_index)
    gib = 1024 ** 3
    return {
        "cuda_available": True,
        "device": torch.cuda.get_device_name(device_index),
        "free_gib": free_b / gib,
        "total_gib": total_b / gib,
        "allocated_gib": torch.cuda.memory_allocated(device_index) / gib,
        "reserved_gib": torch.cuda.memory_reserved(device_index) / gib,
        "max_allocated_gib": torch.cuda.max_memory_allocated(device_index) / gib,
        "free_ratio": free_b / total_b,
    }

def print_cuda_memory(label, device_index=0):
    info = cuda_memory_snapshot(device_index)
    print(f"[CUDA MEMORY] {label}: {info}")
    return info

def cleanup_cuda_memory(device_index=0, reset_peak_stats=True):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize(device_index)
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        if reset_peak_stats:
            torch.cuda.reset_peak_memory_stats(device_index)
        time.sleep(0.5)
        torch.cuda.synchronize(device_index)
    return print_cuda_memory("after cleanup", device_index)

def ensure_cuda_memory_available(
    minimum_free_gib,
    minimum_free_ratio=0.50,
    device_index=0,
):
    print_cuda_memory("before cleanup", device_index)
    info = cleanup_cuda_memory(device_index)
    if not info.get("cuda_available"):
        raise RuntimeError("CUDA is required.")
    if (
        info["free_gib"] < float(minimum_free_gib)
        or info["free_ratio"] < float(minimum_free_ratio)
    ):
        raise RuntimeError(
            "Insufficient free CUDA memory. "
            f"Need >= {minimum_free_gib:.2f} GiB and "
            f">= {minimum_free_ratio:.0%} free, but found "
            f"{info['free_gib']:.2f} GiB and {info['free_ratio']:.0%}. "
            "This notebook cannot free memory owned by another process."
        )
    return info

def delete_globals_and_cleanup(*names):
    namespace = globals()
    for name in names:
        if name in namespace:
            try:
                del namespace[name]
                print("Deleted:", name)
            except Exception as error:
                print(f"Could not delete {name}: {error}")
    return cleanup_cuda_memory()


### System RAM guard

In [5]:
import gc
import os
import time
import ctypes

try:
    import psutil
except ImportError as error:
    raise RuntimeError(
        "psutil is required for system RAM monitoring. "
        "Install it with: pip install psutil"
    ) from error


def system_memory_snapshot():
    """Return system and current-process RAM information in GiB."""
    vm = psutil.virtual_memory()
    process = psutil.Process(os.getpid())
    process_info = process.memory_info()
    gib = 1024 ** 3

    return {
        "total_gib": vm.total / gib,
        "available_gib": vm.available / gib,
        "used_gib": vm.used / gib,
        "available_ratio": vm.available / vm.total,
        "percent_used": float(vm.percent),
        "process_rss_gib": process_info.rss / gib,
        "process_vms_gib": process_info.vms / gib,
    }


def print_system_memory(label):
    info = system_memory_snapshot()
    print(f"[SYSTEM RAM] {label}: {info}")
    return info


def _malloc_trim_linux():
    """Ask glibc to return releasable heap pages to the OS when available."""
    if os.name != "posix":
        return False

    try:
        libc = ctypes.CDLL("libc.so.6")
        result = libc.malloc_trim(0)
        return bool(result)
    except Exception:
        return False


def cleanup_system_memory(wait_seconds=0.5):
    """Collect unreachable Python objects and return free heap pages to Linux.

    This cannot free:
    - objects that still have live Python references,
    - RAM owned by another process,
    - OS file cache on demand without elevated privileges.
    """
    collected = gc.collect()
    trimmed = _malloc_trim_linux()

    if wait_seconds > 0:
        time.sleep(wait_seconds)

    info = print_system_memory("after RAM cleanup")
    info["gc_collected_objects"] = int(collected)
    info["malloc_trim_called"] = bool(trimmed)
    return info


def ensure_system_memory_available(
    minimum_available_gib,
    minimum_available_ratio=0.20,
):
    print_system_memory("before RAM cleanup")
    info = cleanup_system_memory()

    if (
        info["available_gib"] < float(minimum_available_gib)
        or info["available_ratio"] < float(minimum_available_ratio)
    ):
        raise RuntimeError(
            "Insufficient system RAM before run. "
            f"Need >= {minimum_available_gib:.2f} GiB and "
            f">= {minimum_available_ratio:.0%} available, but found "
            f"{info['available_gib']:.2f} GiB and "
            f"{info['available_ratio']:.0%}. "
            "Delete live objects, stop other processes, or restart the kernel."
        )

    return info


def cleanup_all_memory(
    *,
    cuda_device_index=0,
    reset_cuda_peak_stats=True,
):
    """Cleanup current-process system RAM and CUDA cache."""
    ram_info = cleanup_system_memory()
    cuda_info = cleanup_cuda_memory(
        device_index=cuda_device_index,
        reset_peak_stats=reset_cuda_peak_stats,
    )
    return {
        "system_ram": ram_info,
        "cuda": cuda_info,
    }


In [6]:
# Dynamic VRAM policy for V100 / 3090 / higher GPUs.
# The guard scales with total VRAM instead of assuming a 24 GiB RTX 3090.
initial_memory = cuda_memory_snapshot()

if not initial_memory.get("cuda_available"):
    raise RuntimeError("CUDA is required.")

TOTAL_VRAM_GIB = float(initial_memory["total_gib"])

# Keep enough free memory for model loading and the first training batch.
# Examples:
# - V100 16 GB  -> require about 10 GB free
# - V100 32 GB  -> require about 18 GB free
# - RTX 3090    -> require about 14 GB free
MINIMUM_FREE_VRAM_GIB = max(
    8.0,
    min(
        18.0,
        0.58 * TOTAL_VRAM_GIB,
    ),
)
MINIMUM_FREE_VRAM_RATIO = 0.55

print({
    "gpu": initial_memory.get("device"),
    "total_vram_gib": round(TOTAL_VRAM_GIB, 2),
    "minimum_free_vram_gib": round(MINIMUM_FREE_VRAM_GIB, 2),
    "minimum_free_vram_ratio": MINIMUM_FREE_VRAM_RATIO,
})

pre_run_memory = ensure_cuda_memory_available(
    MINIMUM_FREE_VRAM_GIB,
    MINIMUM_FREE_VRAM_RATIO,
)


initial_ram = system_memory_snapshot()
TOTAL_RAM_GIB = float(initial_ram["total_gib"])

# Dynamic RAM threshold:
# require at least 8 GiB, or 20% of total RAM, capped at 32 GiB.
MINIMUM_AVAILABLE_RAM_GIB = max(
    8.0,
    min(
        32.0,
        0.20 * TOTAL_RAM_GIB,
    ),
)
MINIMUM_AVAILABLE_RAM_RATIO = 0.20

print({
    "total_ram_gib": round(TOTAL_RAM_GIB, 2),
    "minimum_available_ram_gib": round(
        MINIMUM_AVAILABLE_RAM_GIB,
        2,
    ),
    "minimum_available_ram_ratio": MINIMUM_AVAILABLE_RAM_RATIO,
})

pre_run_ram = ensure_system_memory_available(
    MINIMUM_AVAILABLE_RAM_GIB,
    MINIMUM_AVAILABLE_RAM_RATIO,
)


{'gpu': 'Tesla V100-SXM2-32GB', 'total_vram_gib': 31.73, 'minimum_free_vram_gib': 18.0, 'minimum_free_vram_ratio': 0.55}
[CUDA MEMORY] before cleanup: {'cuda_available': True, 'device': 'Tesla V100-SXM2-32GB', 'free_gib': 31.12744140625, 'total_gib': 31.7325439453125, 'allocated_gib': 0.0, 'reserved_gib': 0.0, 'max_allocated_gib': 0.0, 'free_ratio': 0.9809311683265821}


[CUDA MEMORY] after cleanup: {'cuda_available': True, 'device': 'Tesla V100-SXM2-32GB', 'free_gib': 31.12744140625, 'total_gib': 31.7325439453125, 'allocated_gib': 0.0, 'reserved_gib': 0.0, 'max_allocated_gib': 0.0, 'free_ratio': 0.9809311683265821}
{'total_ram_gib': 94.0, 'minimum_available_ram_gib': 18.8, 'minimum_available_ram_ratio': 0.2}
[SYSTEM RAM] before RAM cleanup: {'total_gib': 94.0, 'available_gib': 87.7750129699707, 'used_gib': 6.224987030029297, 'available_ratio': 0.9337767337230926, 'percent_used': 6.6, 'process_rss_gib': 0.8451652526855469, 'process_vms_gib': 13.580451965332031}


[SYSTEM RAM] after RAM cleanup: {'total_gib': 94.0, 'available_gib': 87.7747573852539, 'used_gib': 6.225242614746094, 'available_ratio': 0.9337740147367437, 'percent_used': 6.6, 'process_rss_gib': 0.8451652526855469, 'process_vms_gib': 13.58056640625}


## 2. Load shared RoadBuddy helpers

Notebook Phase 2 tái sử dụng đúng các helper đã pass Phase 1. Không copy-paste
một preprocessing pipeline khác.


In [7]:
import sys

sys.path.insert(0, str(Path("./src").resolve()))

from conversation_utils import (
    expand_image_context_for_training,
    require_chat_capability,
)
from inference_debug import (
    inspect_runtime_api,
    strict_multimodal_generate,
)
from label_masking import (
    pad_masked_batch,
    tokenize_and_mask_answer,
)
from tokenizer_utils import (
    get_img_context_token_diagnostics,
    load_vintern_tokenizer,
    validate_img_context_token,
    validate_tokenizer_model_checkpoint,
)
from vintern_preprocessing import preprocess_video

print("preprocess_video signature:", inspect.signature(preprocess_video))


preprocess_video signature: (video_path: 'str | Path', *, model: 'Any', input_size: 'int', num_frames: 'int', dtype: 'torch.dtype', device: 'torch.device | str') -> 'tuple[torch.Tensor, list[int]]'


## 3. Phase 2 MVP configuration

In [8]:
MODEL_NAME = "5CD-AI/Vintern-1B-v3_5"
TOKENIZER_NAME = MODEL_NAME

DATA_ROOT = Path("./data")
TRAIN_JSON = DATA_ROOT / "train" / "train.json"
TEST_JSON = DATA_ROOT / "public_test" / "public_test.json"

PHASE1_DIR = Path("./outputs/roadbuddy_phase1_final")
PHASE1_ARTIFACT_DIR = PHASE1_DIR / "artifacts"

PHASE2_ROOT = Path("./outputs/roadbuddy_phase2")
RUN_ID = "phase2_f3_uniform_v1"

RUN_DIR = PHASE2_ROOT / RUN_ID
CONFIG_DIR = RUN_DIR / "configs"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
METRICS_DIR = RUN_DIR / "metrics"
PREDICTION_DIR = RUN_DIR / "predictions"
LOG_DIR = RUN_DIR / "logs"

for directory in (
    CONFIG_DIR,
    CHECKPOINT_DIR,
    METRICS_DIR,
    PREDICTION_DIR,
    LOG_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

# Safe default for the first 5 AM session.
DEBUG_MODE = True
DEBUG_TRAIN_SAMPLES = 256
DEBUG_VAL_SAMPLES = 64

NUM_FRAMES = 3
SAMPLING_STRATEGY = "uniform"

# Explicit visual budget for the multi-frame MVP.
#
# The old helper returned five tiles per frame, resulting in:
#     3 frames x 5 tiles = 15 tiles
#
# This notebook requires at most two tiles per frame:
#     3 frames x 2 tiles = 6 tiles
MAX_TILES_PER_FRAME = 2
MAX_TOTAL_TILES = NUM_FRAMES * MAX_TILES_PER_FRAME
INPUT_SIZE = 448
MAX_LENGTH = 2048
MAX_NEW_TOKENS = 8

LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 1e-4
NUM_TRAIN_EPOCHS = 1

PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

FP16 = True
BF16 = False
GRADIENT_CHECKPOINTING = False

RUN_STATUS = {
    "phase1_split_loaded": False,
    "model_loaded": False,
    "multiframe_smoke_passed": False,
    "dataset_probe_passed": False,
    "collator_probe_passed": False,
    "lora_ready": False,
    "training_completed": False,
    "lora_updated": False,
    "validation_completed": False,
}

run_config = {
    key: value
    for key, value in {
        "run_id": RUN_ID,
        "seed": SEED,
        "debug_mode": DEBUG_MODE,
        "debug_train_samples": DEBUG_TRAIN_SAMPLES,
        "debug_val_samples": DEBUG_VAL_SAMPLES,
        "num_frames": NUM_FRAMES,
        "sampling_strategy": SAMPLING_STRATEGY,
        "max_tiles_per_frame": MAX_TILES_PER_FRAME,
        "max_total_tiles": MAX_TOTAL_TILES,
        "input_size": INPUT_SIZE,
        "max_length": MAX_LENGTH,
        "max_new_tokens": MAX_NEW_TOKENS,
        "lora_rank": LORA_RANK,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "learning_rate": LEARNING_RATE,
        "epochs": NUM_TRAIN_EPOCHS,
    }.items()
}

(CONFIG_DIR / f"{RUN_ID}.json").write_text(
    json.dumps(run_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(run_config, ensure_ascii=False, indent=2))


{
  "run_id": "phase2_f3_uniform_v1",
  "seed": 42,
  "debug_mode": true,
  "debug_train_samples": 256,
  "debug_val_samples": 64,
  "num_frames": 3,
  "sampling_strategy": "uniform",
  "max_tiles_per_frame": 2,
  "max_total_tiles": 6,
  "input_size": 448,
  "max_length": 2048,
  "max_new_tokens": 8,
  "lora_rank": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "learning_rate": 0.0001,
  "epochs": 1
}


## 4. Annotation resolver và Phase 1 split reuse

Phase 2 không tạo split mới. Nó đọc đúng `train_video_ids.json` và
`validation_video_ids.json` của Phase 1.


In [9]:
def read_annotation_file(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)

    payload = json.loads(path.read_text(encoding="utf-8"))

    if isinstance(payload, list):
        return payload

    if isinstance(payload, dict):
        for key in ("data", "annotations", "samples", "items"):
            if isinstance(payload.get(key), list):
                return payload[key]

    raise ValueError(f"Unsupported annotation schema: {path}")


def first_non_none(mapping, *keys):
    for key in keys:
        if key in mapping and mapping[key] is not None:
            return mapping[key]
    return None


def resolve_video_path(video_value, data_root):
    data_root = Path(data_root)
    raw_path = Path(str(video_value))

    candidates = (
        [raw_path]
        if raw_path.is_absolute()
        else [
            data_root / raw_path,
            raw_path,
            data_root / "train" / "videos" / raw_path.name,
            data_root / "public_test" / "videos" / raw_path.name,
        ]
    )

    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()

    raise FileNotFoundError(
        f"Video not found: {video_value!r}; candidates={list(map(str, candidates))}"
    )


def normalize_record(raw, require_answer=True):
    question = first_non_none(raw, "question", "query")
    answer = first_non_none(raw, "answer", "label")
    video_value = first_non_none(raw, "video_path", "video", "video_id")
    sample_id = first_non_none(raw, "id", "question_id", "uid")

    if question is None or video_value is None:
        raise ValueError(f"Invalid annotation: {raw}")

    if require_answer and answer is None:
        raise ValueError(f"Missing answer: {raw}")

    video_path = resolve_video_path(video_value, DATA_ROOT)

    return {
        "id": str(sample_id if sample_id is not None else video_value),
        "video_id": str(raw.get("video_id", video_path.stem)),
        "video_path": str(video_path),
        "question": str(question).strip(),
        "answer": None if answer is None else str(answer).strip(),
        "support_frames": raw.get("support_frames", []),
    }


required_split_files = {
    "train": PHASE1_ARTIFACT_DIR / "train_video_ids.json",
    "validation": PHASE1_ARTIFACT_DIR / "validation_video_ids.json",
}

for split_name, split_path in required_split_files.items():
    if not split_path.is_file():
        raise FileNotFoundError(
            f"Missing Phase 1 {split_name} split artifact: {split_path}"
        )

train_video_ids = set(json.loads(
    required_split_files["train"].read_text(encoding="utf-8")
))
validation_video_ids = set(json.loads(
    required_split_files["validation"].read_text(encoding="utf-8")
))

assert train_video_ids.isdisjoint(validation_video_ids)

all_records = [
    normalize_record(raw, require_answer=True)
    for raw in read_annotation_file(TRAIN_JSON)
]

train_records = [
    row for row in all_records
    if row["video_id"] in train_video_ids
]
val_records = [
    row for row in all_records
    if row["video_id"] in validation_video_ids
]

if not train_records or not val_records:
    raise RuntimeError("Phase 1 split could not be reconstructed.")

assert {
    row["video_id"] for row in train_records
}.isdisjoint({
    row["video_id"] for row in val_records
})

RUN_STATUS["phase1_split_loaded"] = True

print("Train samples:", len(train_records))
print("Validation samples:", len(val_records))
print("Train videos:", len({x['video_id'] for x in train_records}))
print("Validation videos:", len({x['video_id'] for x in val_records}))


Train samples: 1171
Validation samples: 319
Train videos: 439
Validation videos: 110


## 5. Load original Vintern model và tokenizer

### Transformers remote-model compatibility

In [10]:
# Compatibility shim for older InternVL/Vintern trust_remote_code models
# running on recent Transformers versions.
#
# Recent Transformers loaders access `model.all_tied_weights_keys` during
# from_pretrained(). Older remote model classes may not initialize it.
print("Transformers version:", transformers.__version__)

if not hasattr(PreTrainedModel, "all_tied_weights_keys"):
    PreTrainedModel.all_tied_weights_keys = {}
    print(
        "Applied compatibility shim: "
        "PreTrainedModel.all_tied_weights_keys = {}"
    )
else:
    print(
        "No tied-weights shim required; "
        "PreTrainedModel already exposes all_tied_weights_keys."
    )

Transformers version: 5.14.1
Applied compatibility shim: PreTrainedModel.all_tied_weights_keys = {}


In [11]:
validate_tokenizer_model_checkpoint(
    model_name_or_path=MODEL_NAME,
    tokenizer_name_or_path=TOKENIZER_NAME,
)

tokenizer = load_vintern_tokenizer(TOKENIZER_NAME)

try:
    model = AutoModel.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        use_flash_attn=False,
    )
except AttributeError as error:
    if "all_tied_weights_keys" in str(error):
        raise RuntimeError(
            "Vintern remote model code is incompatible with the currently "
            f"installed Transformers {transformers.__version__}. "
            "The compatibility shim was applied but loading still failed. "
            "Use the same Transformers 4.x version recorded by the successful "
            "Phase 1 environment, then restart the kernel."
        ) from error
    raise
model = model.eval().to(device)

img_context_diagnostics = get_img_context_token_diagnostics(tokenizer)
img_context_token_id = validate_img_context_token(
    tokenizer=tokenizer,
    model=model,
)
model.img_context_token_id = img_context_token_id

inspect_runtime_api(model, tokenizer)
conversation_route = require_chat_capability(model, tokenizer)

RUN_STATUS["model_loaded"] = True

print(json.dumps(img_context_diagnostics, ensure_ascii=False, indent=2))
print("Conversation route:", conversation_route)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


/opt/venvs/vintern-py31213/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.


Loading weights:   0%|          | 0/637 [00:00<?, ?it/s]

<class 'transformers_modules._5CD_hyphen_AI.Vintern_hyphen_1B_hyphen_v3_5.b98f263eab246eb5269ade64edbdca8a887dc44d.modeling_internvl_chat.InternVLChatModel'>
InternVLChatConfig {
  "architectures": [
    "InternVLChatModel"
  ],
  "auto_map": {
    "AutoConfig": "5CD-AI/Vintern-1B-v3_5--configuration_internvl_chat.InternVLChatConfig",
    "AutoModel": "5CD-AI/Vintern-1B-v3_5--modeling_internvl_chat.InternVLChatModel",
    "AutoModelForCausalLM": "5CD-AI/Vintern-1B-v3_5--modeling_internvl_chat.InternVLChatModel"
  },
  "downsample_ratio": 0.5,
  "dtype": "float16",
  "dynamic_image_size": true,
  "force_image_size": 448,
  "llm_config": {
    "_name_or_path": "Qwen/Qwen2.5-0.5B-Instruct",
    "add_cross_attention": false,
    "architectures": [
      "Qwen2ForCausalLM"
    ],
    "attention_dropout": 0.0,
    "bos_token_id": 151643,
    "cross_attention_hidden_size": null,
    "decoder_start_token_id": null,
    "dtype": "bfloat16",
    "eos_token_id": 151645,
    "finetuning_task": nul

## 6. Multi-frame preprocessing wrapper và guards

Helper Phase 1 đã có `num_frames`. Wrapper dưới đây chỉ thêm logging và
tile-budget guards; nó không thay đổi pixel normalization.


In [12]:
def _select_tile_indices(tile_count, max_tiles):
    """Select a deterministic subset of tiles from one frame.

    InternVL-style dynamic preprocessing commonly places spatial crops first
    and an optional thumbnail last. For a two-tile budget, keeping the first
    and last tiles preserves one detailed crop and one global view.

    The function remains generic if a future helper returns fewer tiles.
    """
    if tile_count <= 0:
        raise ValueError(f"tile_count must be positive, got {tile_count}.")

    if max_tiles <= 0:
        raise ValueError(f"max_tiles must be positive, got {max_tiles}.")

    if tile_count <= max_tiles:
        return list(range(tile_count))

    if max_tiles == 1:
        # Prefer the final tile because InternVL dynamic preprocessing often
        # appends a thumbnail/global view at the end.
        return [tile_count - 1]

    # Spread selections across the full tile sequence.
    indices = np.linspace(
        0,
        tile_count - 1,
        num=max_tiles,
        dtype=int,
    ).tolist()

    # np.linspace can theoretically repeat indices for tiny ranges.
    unique_indices = []
    for index in indices:
        if index not in unique_indices:
            unique_indices.append(index)

    if len(unique_indices) != max_tiles:
        raise RuntimeError(
            f"Could not select {max_tiles} unique tiles from {tile_count}."
        )

    return unique_indices


def _cap_tiles_per_frame(
    pixel_values,
    num_patches_list,
    max_tiles_per_frame,
):
    """Reduce each frame's tile group without changing image normalization.

    The original Phase-1 helper is still responsible for:
      - video decoding,
      - frame sampling,
      - dynamic image crops,
      - image normalization,
      - dtype and device placement.

    This function only selects a deterministic subset from each frame group.
    """
    if not torch.is_tensor(pixel_values):
        raise TypeError(
            f"pixel_values must be a tensor, got {type(pixel_values)}."
        )

    patch_counts = [int(value) for value in num_patches_list]

    if sum(patch_counts) != int(pixel_values.shape[0]):
        raise RuntimeError(
            "Input patch alignment failed before tile capping: "
            f"sum(counts)={sum(patch_counts)}, "
            f"tensor tiles={pixel_values.shape[0]}."
        )

    selected_groups = []
    capped_counts = []
    selection_log = []

    offset = 0

    for frame_index, original_count in enumerate(patch_counts):
        frame_tiles = pixel_values[
            offset : offset + original_count
        ]

        local_indices = _select_tile_indices(
            tile_count=original_count,
            max_tiles=max_tiles_per_frame,
        )

        selected_groups.append(
            frame_tiles[local_indices]
        )
        capped_counts.append(len(local_indices))

        selection_log.append({
            "frame_index": frame_index,
            "original_count": original_count,
            "selected_local_indices": local_indices,
            "selected_count": len(local_indices),
        })

        offset += original_count

    capped_pixel_values = torch.cat(
        selected_groups,
        dim=0,
    )

    if sum(capped_counts) != int(capped_pixel_values.shape[0]):
        raise RuntimeError(
            "Output patch alignment failed after tile capping."
        )

    return (
        capped_pixel_values,
        capped_counts,
        selection_log,
    )


def preprocess_multiframe(record, target_model, dtype=torch.float16):
    """Preprocess 3 frames and locally cap each frame to two visual tiles.

    Why local capping is needed
    ---------------------------
    The installed Phase-1 helper has this fixed signature:

        preprocess_video(
            video_path,
            *,
            model,
            input_size,
            num_frames,
            dtype,
            device,
        )

    It does not expose a tile-limit parameter. Calling it with three frames
    returns five tiles per frame, or 15 tiles total.

    Instead of modifying src/vintern_preprocessing.py, this notebook:
      1. Calls the validated Phase-1 helper unchanged.
      2. Receives the original groups, e.g. [5, 5, 5].
      3. Selects at most two tiles from every group.
      4. Returns capped groups, e.g. [2, 2, 2].

    This keeps the exact Phase-1 image normalization while respecting the
    Phase-2 visual-token budget.
    """
    result = preprocess_video(
        record["video_path"],
        model=target_model,
        input_size=INPUT_SIZE,
        num_frames=NUM_FRAMES,
        dtype=dtype,
        device=device,
    )

    if not isinstance(result, tuple):
        raise TypeError(
            f"Unexpected preprocess_video result: {type(result)}."
        )

    if len(result) == 2:
        original_pixels, original_patch_counts = result
        selected_timestamps = None
    elif len(result) == 3:
        (
            original_pixels,
            original_patch_counts,
            selected_timestamps,
        ) = result
    else:
        raise ValueError(
            f"Expected 2 or 3 outputs, got {len(result)}."
        )

    original_patch_counts = [
        int(value)
        for value in original_patch_counts
    ]

    if len(original_patch_counts) != NUM_FRAMES:
        raise RuntimeError(
            f"Expected {NUM_FRAMES} frame groups, received "
            f"{original_patch_counts}."
        )

    (
        pixel_values,
        num_patches_list,
        tile_selection_log,
    ) = _cap_tiles_per_frame(
        pixel_values=original_pixels,
        num_patches_list=original_patch_counts,
        max_tiles_per_frame=MAX_TILES_PER_FRAME,
    )

    total_tiles = sum(num_patches_list)

    if any(
        count > MAX_TILES_PER_FRAME
        for count in num_patches_list
    ):
        raise RuntimeError(
            f"Per-frame budget failed: {num_patches_list}."
        )

    if total_tiles > MAX_TOTAL_TILES:
        raise RuntimeError(
            f"Visual budget exceeded after capping: "
            f"{total_tiles} > {MAX_TOTAL_TILES}."
        )

    if total_tiles != int(pixel_values.shape[0]):
        raise RuntimeError(
            "Patch alignment failed after local tile capping."
        )

    print(
        "Local multi-frame tile cap:",
        {
            "original_patch_counts": original_patch_counts,
            "capped_patch_counts": num_patches_list,
            "selection_log": tile_selection_log,
            "total_tiles": total_tiles,
            "total_budget": MAX_TOTAL_TILES,
        },
    )

    return (
        pixel_values,
        num_patches_list,
        selected_timestamps,
    )


In [13]:
def build_multiframe_question(question, num_patches_list):
    """Build a Vintern question with one <image> placeholder per frame.

    strict_multimodal_generate() validates that the number of image placeholders
    matches len(num_patches_list). For three sampled frames, the prompt must
    contain exactly three <image> placeholders.

    Example:
        <image>
        <image>
        <image>
        Câu hỏi...
    """
    if question is None:
        raise ValueError("question must not be None.")

    patch_counts = [int(value) for value in num_patches_list]

    if not patch_counts:
        raise ValueError("num_patches_list must not be empty.")

    if any(value <= 0 for value in patch_counts):
        raise ValueError(
            f"All patch counts must be positive: {patch_counts}"
        )

    # Remove existing placeholders so the function is idempotent.
    clean_question = str(question).replace("<image>", "").strip()

    image_prefix = "\n".join(
        "<image>"
        for _ in patch_counts
    )

    multimodal_question = (
        f"{image_prefix}\n{clean_question}"
    )

    actual_count = multimodal_question.count("<image>")
    expected_count = len(patch_counts)

    if actual_count != expected_count:
        raise RuntimeError(
            "Failed to construct the multi-frame prompt: "
            f"expected {expected_count} placeholders, "
            f"found {actual_count}."
        )

    return multimodal_question


def validate_multiframe_prompt(question, num_patches_list):
    """Validate prompt/image-group alignment before model.chat()."""
    expected_count = len(num_patches_list)
    actual_count = str(question).count("<image>")

    if actual_count != expected_count:
        raise ValueError(
            "Question image placeholder count does not match "
            f"num_patches_list: expected {expected_count}, "
            f"found {actual_count}."
        )

    return question


In [14]:
def build_training_question(
    question,
    num_patches_list,
    tokenizer,
    model,
):
    """Build the training prompt using the validated Phase-1 API.

    Important distinction
    ---------------------
    Inference through model.chat()/strict_multimodal_generate() requires one
    raw <image> placeholder for each entry in num_patches_list.

    Training through model.forward() uses the Phase-1 helper, which accepts the
    original text question and inserts the image-context block itself. It must
    not receive a prompt that already contains raw <image> placeholders.

    All capped frame tiles are concatenated in pixel_values, so the training
    image-context length is based on the total number of tiles.
    """
    patch_counts = [int(value) for value in num_patches_list]

    if not patch_counts:
        raise ValueError("num_patches_list must not be empty.")

    if any(value <= 0 for value in patch_counts):
        raise ValueError(
            f"All patch counts must be positive: {patch_counts}"
        )

    clean_question = str(question).replace("<image>", "").strip()

    if not clean_question:
        raise ValueError("Training question is empty.")

    total_patch_count = sum(patch_counts)

    training_question = expand_image_context_for_training(
        clean_question,
        tokenizer=tokenizer,
        model=model,
        patch_count=total_patch_count,
    )

    if not isinstance(training_question, str):
        raise TypeError(
            "expand_image_context_for_training() must return a string, "
            f"got {type(training_question)}."
        )

    # The helper may preserve an outer <image> wrapper depending on the source
    # implementation; therefore we do not incorrectly require zero raw tokens.
    # What matters is that image-context tokens were inserted and the result is
    # longer than the original question.
    if len(training_question) <= len(clean_question):
        raise RuntimeError(
            "Training image context was not expanded: "
            f"original length={len(clean_question)}, "
            f"expanded length={len(training_question)}."
        )

    return training_question


### Multi-frame prompt alignment

`num_patches_list` describes one visual group per sampled frame. Therefore:

```text
num_patches_list = [2, 2, 2]
```

requires:

```text
<image>
<image>
<image>
Câu hỏi...
```

A single `<image>` placeholder is valid only when `num_patches_list` contains
one group. The notebook now builds the prompt automatically for smoke testing,
training and validation.


### Local tile-cap strategy

`src/vintern_preprocessing.py` không có tham số giới hạn dynamic tiles. Bản này
không yêu cầu sửa helper. Nó gọi preprocessing Phase 1 như cũ, sau đó giảm từng
nhóm frame từ `[5, 5, 5]` xuống tối đa `[2, 2, 2]`.

Với hai tiles, notebook chọn vị trí đầu và cuối của mỗi nhóm. Cách này thường
giữ được một crop chi tiết và thumbnail/global tile cuối của InternVL.

Các guard tiếp tục xác minh:

- Số nhóm patch bằng `NUM_FRAMES`.
- Tổng patch khớp `pixel_values.shape[0]`.
- Mỗi frame không vượt `MAX_TILES_PER_FRAME`.
- Tổng tiles không vượt `MAX_TOTAL_TILES`.


## 7. Original-model 3-frame smoke test

In [15]:
smoke_record = val_records[0]

smoke_pixels, smoke_patch_counts, smoke_timestamps = preprocess_multiframe(
    smoke_record,
    target_model=model,
)

smoke_question = build_multiframe_question(
    smoke_record["question"],
    smoke_patch_counts,
)
validate_multiframe_prompt(
    smoke_question,
    smoke_patch_counts,
)

with torch.inference_mode():
    smoke_result = strict_multimodal_generate(
        model=model,
        tokenizer=tokenizer,
        question=smoke_question,
        pixel_values=smoke_pixels,
        num_patches_list=smoke_patch_counts,
        max_new_tokens=MAX_NEW_TOKENS,
    )

smoke_response = str(smoke_result["response"]).strip()

print("Video:", smoke_record["video_path"])
print("Original question:", smoke_record["question"])
print("Multiframe prompt:", repr(smoke_question))
print("Image placeholders:", smoke_question.count("<image>"))
print("Pixel shape:", tuple(smoke_pixels.shape))
print("Patch counts:", smoke_patch_counts)
print("Selected timestamps:", smoke_timestamps)
print("Route:", smoke_result.get("route"))
print("Response:", repr(smoke_response))

if not smoke_response:
    raise RuntimeError("3-frame original-model smoke test returned empty output.")

RUN_STATUS["multiframe_smoke_passed"] = True


Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}
Validated <IMG_CONTEXT> token ID: 151667


Video: /workspace/RoadBuddy/data/train/videos/2b840c67_386_clip_002_0008_0018_Y.mp4
Original question: Nếu xe ô tô đang chạy ở làn ngoài cùng bên phải trong video này thì xe đó chỉ được phép rẽ phải?
Multiframe prompt: '<image>\n<image>\n<image>\nNếu xe ô tô đang chạy ở làn ngoài cùng bên phải trong video này thì xe đó chỉ được phép rẽ phải?'
Image placeholders: 3
Pixel shape: (6, 3, 448, 448)
Patch counts: [2, 2, 2]
Selected timestamps: None
Route: model.chat
Response: 'Không, xe ô tô đang chạy ở'


## 8. Dataset và token-level masking

In [16]:
def validate_image_context_alignment(sample, model):
    """Ensure truncation did not remove required multimodal context tokens."""
    context_token_id = getattr(model, "img_context_token_id", None)
    tokens_per_tile = getattr(model, "num_image_token", None)

    if context_token_id is None or tokens_per_tile is None:
        raise RuntimeError(
            "Model does not expose img_context_token_id/num_image_token."
        )

    actual = int(
        (sample["input_ids"] == int(context_token_id)).sum().item()
    )
    expected = int(sample["total_patch_count"]) * int(tokens_per_tile)

    if actual != expected:
        raise RuntimeError(
            "Image-context alignment failed after tokenization/truncation: "
            f"expected {expected}, observed {actual}."
        )

    return {
        "expected_image_context_tokens": expected,
        "actual_image_context_tokens": actual,
    }


### Training versus inference prompt routing

Hai đường chạy dùng prompt khác nhau:

**Inference (`model.chat`)**

```text
<image>
<image>
<image>
Câu hỏi...
```

Số placeholder phải bằng `len(num_patches_list)`.

**Training (`model.forward`)**

```python
training_question = expand_image_context_for_training(
    original_question,
    tokenizer=tokenizer,
    model=model,
    patch_count=sum(num_patches_list),
)
```

Helper Phase 1 tự tạo image-context block. Không truyền prompt đã có nhiều
`<image>` vào helper này. Việc phân nhóm frame vẫn được giữ trong
`num_patches_list`, còn số image-context token được tính từ tổng tiles đã cap.


### Answer-preserving label masking

Multi-frame visual context can exceed the prompt token budget. This wrapper
preserves the assistant answer at the end and truncates only the oldest prompt
tokens when necessary.


In [17]:
def tokenize_and_mask_answer_preserving_suffix(
    tokenizer,
    training_question,
    answer,
    max_length,
):
    """Tokenize training text while guaranteeing answer supervision.

    Why this wrapper exists
    -----------------------
    Multi-frame image context can consume most of MAX_LENGTH. A naive
    right-truncation of the complete conversation can remove the assistant
    answer at the end, leaving no supervised tokens.

    Policy:
      1. Try the validated Phase-1 helper first.
      2. If it reports no assistant tokens, build prompt/answer boundaries
         explicitly with the tokenizer chat template.
      3. If the sequence is too long, truncate the *left side of the prompt*,
         never the answer suffix.
      4. Mask every prompt token with -100 and supervise only answer tokens.
    """
    clean_answer = str(answer).strip()

    if not clean_answer:
        raise ValueError("Answer must not be empty.")

    try:
        encoded = tokenize_and_mask_answer(
            tokenizer,
            training_question,
            clean_answer,
            max_length=max_length,
        )

        supervised = int((encoded["labels"] != -100).sum())

        if supervised > 0:
            encoded["masking_strategy"] = "phase1_helper"
            encoded["prompt_tokens_before_truncation"] = None
            encoded["answer_tokens"] = supervised
            encoded["left_truncated_prompt_tokens"] = 0
            return encoded

    except RuntimeError as error:
        if "no assistant answer tokens" not in str(error).lower():
            raise

    user_messages = [
        {
            "role": "user",
            "content": training_question,
        }
    ]
    full_messages = [
        {
            "role": "user",
            "content": training_question,
        },
        {
            "role": "assistant",
            "content": clean_answer,
        },
    ]

    # Build the exact text produced by the active tokenizer template.
    prompt_text = tokenizer.apply_chat_template(
        user_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = tokenizer.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
    )["input_ids"]
    full_ids = tokenizer(
        full_text,
        add_special_tokens=False,
    )["input_ids"]

    # Prefer the template-derived suffix when full_ids begins with prompt_ids.
    if (
        len(full_ids) > len(prompt_ids)
        and full_ids[: len(prompt_ids)] == prompt_ids
    ):
        answer_ids = full_ids[len(prompt_ids) :]
        boundary_strategy = "chat_template_prefix"
    else:
        # Some remote templates do not produce a strict token-prefix relation.
        # In that case append an explicitly tokenized answer suffix.
        answer_ids = tokenizer(
            clean_answer,
            add_special_tokens=False,
        )["input_ids"]

        eos_token_id = tokenizer.eos_token_id

        if (
            eos_token_id is not None
            and (
                not answer_ids
                or answer_ids[-1] != eos_token_id
            )
        ):
            answer_ids = answer_ids + [int(eos_token_id)]

        boundary_strategy = "explicit_answer_suffix"

    if not answer_ids:
        raise RuntimeError(
            "Could not tokenize any assistant answer tokens."
        )

    # Always preserve answer supervision. If necessary, retain only the last
    # max_length tokens of the answer itself, though A/B/C/D answers are tiny.
    if len(answer_ids) >= max_length:
        kept_answer_ids = answer_ids[-max_length:]
        kept_prompt_ids = []
        truncated_prompt_count = len(prompt_ids)
    else:
        prompt_budget = max_length - len(answer_ids)
        kept_prompt_ids = prompt_ids[-prompt_budget:]
        kept_answer_ids = answer_ids
        truncated_prompt_count = len(prompt_ids) - len(kept_prompt_ids)

    input_ids = kept_prompt_ids + kept_answer_ids
    labels = (
        [-100] * len(kept_prompt_ids)
        + kept_answer_ids.copy()
    )
    attention_mask = [1] * len(input_ids)

    input_ids_tensor = torch.tensor(
        input_ids,
        dtype=torch.long,
    )
    attention_mask_tensor = torch.tensor(
        attention_mask,
        dtype=torch.long,
    )
    labels_tensor = torch.tensor(
        labels,
        dtype=torch.long,
    )

    supervised = int(
        (labels_tensor != -100).sum()
    )

    if supervised <= 0:
        raise RuntimeError(
            "Answer-preserving masking still produced no supervised tokens."
        )

    return {
        "input_ids": input_ids_tensor,
        "attention_mask": attention_mask_tensor,
        "labels": labels_tensor,
        "masking_strategy": (
            "answer_preserving_"
            + boundary_strategy
        ),
        "prompt_tokens_before_truncation": len(prompt_ids),
        "answer_tokens": len(kept_answer_ids),
        "left_truncated_prompt_tokens": truncated_prompt_count,
    }


In [18]:
class RoadBuddyPhase2Dataset(Dataset):
    def __init__(self, records, tokenizer, model):
        self.records = records
        self.tokenizer = tokenizer
        self.model = model

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]

        pixel_values, patch_counts, selected_timestamps = preprocess_multiframe(
            row,
            target_model=self.model,
        )

        # Training and inference use different prompt routes.
        # model.chat() needs one <image> placeholder per frame group, while
        # model.forward() training follows the validated Phase-1 helper and
        # receives the original question plus the total capped patch count.
        prompt = build_training_question(
            question=row["question"],
            num_patches_list=patch_counts,
            tokenizer=self.tokenizer,
            model=self.model,
        )

        # Keep answer handling in the label-masking helper. This matches the
        # validated Phase-1 API and prevents answer tokens from being inserted
        # twice.
        tokenized = tokenize_and_mask_answer_preserving_suffix(
            self.tokenizer,
            prompt,
            row["answer"],
            max_length=MAX_LENGTH,
        )

        supervised_count = int((tokenized["labels"] != -100).sum())

        if supervised_count <= 0:
            raise RuntimeError(
                f"No supervised answer tokens for sample {row['id']}."
            )

        return {
            "input_ids": tokenized["input_ids"],
            "attention_mask": tokenized["attention_mask"],
            "labels": tokenized["labels"],
            "pixel_values": pixel_values.cpu(),
            "image_flags": torch.ones(
                pixel_values.shape[0],
                dtype=torch.long,
            ),
            "num_patches_list": patch_counts,
            "sample_id": row["id"],
            "selected_timestamps": selected_timestamps,
            "expanded_prompt_length": len(prompt),
            "total_patch_count": sum(patch_counts),
            "masking_strategy": tokenized.get(
                "masking_strategy",
                "unknown",
            ),
            "prompt_tokens_before_truncation": tokenized.get(
                "prompt_tokens_before_truncation"
            ),
            "answer_tokens": tokenized.get(
                "answer_tokens",
                supervised_count,
            ),
            "left_truncated_prompt_tokens": tokenized.get(
                "left_truncated_prompt_tokens",
                0,
            ),
        }


train_dataset_full = RoadBuddyPhase2Dataset(
    train_records,
    tokenizer,
    model,
)
val_dataset_full = RoadBuddyPhase2Dataset(
    val_records,
    tokenizer,
    model,
)

if DEBUG_MODE:
    train_limit = min(DEBUG_TRAIN_SAMPLES, len(train_dataset_full))
    val_limit = min(DEBUG_VAL_SAMPLES, len(val_dataset_full))
    train_dataset = Subset(train_dataset_full, range(train_limit))
    val_dataset = Subset(val_dataset_full, range(val_limit))
else:
    train_dataset = train_dataset_full
    val_dataset = val_dataset_full

probe_rows = []

for index in range(min(16, len(train_dataset))):
    sample = train_dataset[index]
    alignment = validate_image_context_alignment(sample, model)
    probe_rows.append({
        "index": index,
        "tiles": int(sample["pixel_values"].shape[0]),
        "expected_image_context_tokens": alignment["expected_image_context_tokens"],
        "actual_image_context_tokens": alignment["actual_image_context_tokens"],
        "patch_groups": sample["num_patches_list"],
        "supervised_tokens": int((sample["labels"] != -100).sum()),
        "total_tokens": int(sample["input_ids"].numel()),
        "expanded_prompt_length": int(sample["expanded_prompt_length"]),
        "total_patch_count": int(sample["total_patch_count"]),
        "masking_strategy": sample["masking_strategy"],
        "prompt_tokens_before_truncation": (
            sample["prompt_tokens_before_truncation"]
        ),
        "answer_tokens": int(sample["answer_tokens"]),
        "left_truncated_prompt_tokens": int(
            sample["left_truncated_prompt_tokens"]
        ),
    })

probe_df = pd.DataFrame(probe_rows)
display(probe_df)

if (probe_df["supervised_tokens"] <= 0).any():
    raise RuntimeError("Dataset masking probe failed.")

if (probe_df["answer_tokens"] <= 0).any():
    raise RuntimeError("Answer token preservation probe failed.")

print(
    "Samples requiring left prompt truncation:",
    int((probe_df["left_truncated_prompt_tokens"] > 0).sum()),
)

RUN_STATUS["dataset_probe_passed"] = True

Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}


Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}


Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}


Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}


Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}


...

Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}


Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}


Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}


,index,tiles,expected_image_context_tokens,actual_image_context_tokens,patch_groups,supervised_tokens,total_tokens,expanded_prompt_length,total_patch_count,masking_strategy,prompt_tokens_before_truncation,answer_tokens,left_truncated_prompt_tokens
0,0,6,1536,1536,"[2, 2, 2]",5,1587,20031,6,answer_preserving_chat_template_prefix,1582,5,0
1,1,6,1536,1536,"[2, 2, 2]",7,1590,20034,6,answer_preserving_chat_template_prefix,1583,7,0
2,2,6,1536,1536,"[2, 2, 2]",5,1594,20045,6,answer_preserving_chat_template_prefix,1589,5,0
3,3,6,1536,1536,"[2, 2, 2]",10,1600,20057,6,answer_preserving_chat_template_prefix,1590,10,0
4,4,6,1536,1536,"[2, 2, 2]",6,1590,20030,6,answer_preserving_chat_template_prefix,1584,6,0
5,5,6,1536,1536,"[2, 2, 2]",5,1593,20050,6,answer_preserving_chat_template_prefix,1588,5,0
6,6,6,1536,1536,"[2, 2, 2]",8,1586,20007,6,answer_preserving_chat_template_prefix,1578,8,0
7,7,6,1536,1536,"[2, 2, 2]",6,1608,20097,6,answer_preserving_chat_template_prefix,1602,6,0
8,8,6,1536,1536,"[2, 2, 2]",25,1615,20056,6,answer_preserving_chat_template_prefix,1590,25,0
9,9,6,1536,1536,"[2, 2, 2]",9,1603,20074,6,answer_preserving_chat_template_prefix,1594,9,0


Samples requiring left prompt truncation: 0


## 9. Multi-modal collator

In [19]:
# Confirm the helper API loaded from src/label_masking.py.
pad_masked_batch_signature = inspect.signature(pad_masked_batch)
print("pad_masked_batch signature:", pad_masked_batch_signature)

parameter_names = list(
    pad_masked_batch_signature.parameters
)

if parameter_names[:2] != ["features", "pad_token_id"]:
    raise RuntimeError(
        "Unexpected pad_masked_batch API. "
        f"Expected ['features', 'pad_token_id'], got {parameter_names}."
    )

print("Tokenizer pad_token_id:", tokenizer.pad_token_id)
print("Tokenizer eos_token_id:", tokenizer.eos_token_id)


pad_masked_batch signature: (features: 'list[dict[str, torch.Tensor]]', pad_token_id: 'int') -> 'dict[str, torch.Tensor]'
Tokenizer pad_token_id: 151643
Tokenizer eos_token_id: 151645


### Section 9 final API fix

Traceback xác nhận source thật có signature:

```python
pad_masked_batch(features, pad_token_id)
```

Ý nghĩa:

- `features`: danh sách sample dictionary từ dataset.
- `pad_token_id`: một số nguyên dùng để pad `input_ids`.

Notebook trước đã truyền tokenizer object ở vị trí đầu, nên helper cố iterate
`Qwen2Tokenizer` như một danh sách sample và phát sinh:

```text
TypeError: 'Qwen2Tokenizer' object is not iterable
```

Bản sửa gọi:

```python
pad_masked_batch(
    features,
    tokenizer.pad_token_id,
)
```

Nếu tokenizer không có `pad_token_id`, notebook dùng `eos_token_id` làm fallback.


In [20]:
class RoadBuddyMultiframeCollator:
    ALLOWED_MODEL_KEYS = {
        "input_ids",
        "attention_mask",
        "labels",
        "pixel_values",
        "image_flags",
    }

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        # Actual source signature:
        #     pad_masked_batch(features, pad_token_id)
        # The first argument must be the list of sample dictionaries.
        # The second argument must be an integer token ID, not the tokenizer.
        pad_token_id = self.tokenizer.pad_token_id

        if pad_token_id is None:
            pad_token_id = self.tokenizer.eos_token_id

        if pad_token_id is None:
            raise RuntimeError(
                "Tokenizer has neither pad_token_id nor eos_token_id."
            )

        text_batch = pad_masked_batch(
            features,
            int(pad_token_id),
        )

        pixel_values = torch.cat(
            [feature["pixel_values"] for feature in features],
            dim=0,
        )
        image_flags = torch.cat(
            [feature["image_flags"] for feature in features],
            dim=0,
        )
        num_patches_list = [
            patch_count
            for feature in features
            for patch_count in feature["num_patches_list"]
        ]

        # num_patches_list is useful for collator-side validation and
        # inference through model.chat(), but InternVLChatModel.forward()
        # does not accept it. Therefore it must not be included in the batch
        # returned to Trainer.
        batch = {
            **text_batch,
            "pixel_values": pixel_values,
            "image_flags": image_flags,
        }

        batch.pop("inputs_embeds", None)

        unexpected = set(batch) - self.ALLOWED_MODEL_KEYS
        if unexpected:
            raise RuntimeError(f"Unexpected model batch keys: {unexpected}")

        if sum(num_patches_list) != pixel_values.shape[0]:
            raise RuntimeError("Collator patch alignment failed.")

        if int((batch["labels"] != -100).sum()) <= 0:
            raise RuntimeError("Collated batch has no supervised tokens.")

        return batch


data_collator = RoadBuddyMultiframeCollator(tokenizer)

collator_probe = data_collator([
    train_dataset[0],
    train_dataset[1],
])

print({
    key: (
        tuple(value.shape)
        if torch.is_tensor(value)
        else value
    )
    for key, value in collator_probe.items()
})

print(
    "Expected patch groups for two samples:",
    [
        patch_count
        for sample_index in range(2)
        for patch_count in train_dataset[sample_index]["num_patches_list"]
    ],
)

RUN_STATUS["collator_probe_passed"] = True


Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}


Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}
{'input_ids': (2, 1590), 'attention_mask': (2, 1590), 'labels': (2, 1590), 'pixel_values': (12, 3, 448, 448), 'image_flags': (12,)}


Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}


Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}
Expected patch groups for two samples: [2, 2, 2, 2, 2, 2]


## 10. LoRA setup: FP16 frozen base + FP32 adapters

In [21]:
if not all([
    RUN_STATUS["multiframe_smoke_passed"],
    RUN_STATUS["dataset_probe_passed"],
    RUN_STATUS["collator_probe_passed"],
]):
    raise RuntimeError("All Phase 2 probes must pass before LoRA setup.")

for parameter in model.parameters():
    parameter.requires_grad = False

linear_suffixes = {
    name.rsplit(".", 1)[-1]
    for name, module in model.named_modules()
    if isinstance(module, torch.nn.Linear)
    and not any(x in name.lower() for x in ("vision", "visual", "vit"))
}

target_modules = [
    name
    for name in ("q_proj", "k_proj", "v_proj", "o_proj")
    if name in linear_suffixes
]

if not target_modules:
    raise RuntimeError(
        f"No safe LoRA targets found. Linear suffixes: {sorted(linear_suffixes)}"
    )

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=target_modules,
)

model = get_peft_model(model, lora_config)

if "CausalLM" in type(model).__name__:
    raise RuntimeError(
        f"Incorrect PEFT wrapper: {type(model).__name__}. "
        "Do not set task_type=CAUSAL_LM."
    )

for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()

invalid_trainables = [
    (name, str(parameter.dtype))
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
    and (
        "lora_" not in name.lower()
        or parameter.dtype != torch.float32
    )
]

if invalid_trainables:
    raise RuntimeError(f"Invalid trainable parameters: {invalid_trainables[:10]}")

for candidate in (
    model,
    getattr(model, "base_model", None),
    getattr(getattr(model, "base_model", None), "model", None),
):
    config = getattr(candidate, "config", None)
    if config is not None and hasattr(config, "use_cache"):
        config.use_cache = False

initial_lora_state = {
    name: parameter.detach().cpu().clone()
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
}

RUN_STATUS["lora_ready"] = True

print("PEFT class:", type(model).__name__)
print("LoRA targets:", target_modules)
model.print_trainable_parameters()


PEFT class: PeftModel
LoRA targets: ['q_proj', 'k_proj', 'v_proj', 'o_proj']
trainable params: 2,162,688 || all params: 940,355,712 || trainable%: 0.2300


## 11. Train MVP

In [22]:
# Audit the actual forward signature before Trainer sends any batch.
forward_signature = inspect.signature(model.forward)
forward_parameter_names = set(forward_signature.parameters)

print("model.forward signature:", forward_signature)
print("Trainer batch keys:", sorted(data_collator.ALLOWED_MODEL_KEYS))

if "num_patches_list" in forward_parameter_names:
    print("Note: forward unexpectedly supports num_patches_list.")
else:
    print(
        "Confirmed: num_patches_list is inference/collator metadata only "
        "and will not be passed to model.forward()."
    )

required_forward_keys = {
    "input_ids",
    "attention_mask",
    "labels",
    "pixel_values",
    "image_flags",
}

missing_forward_keys = [
    key
    for key in required_forward_keys
    if key not in forward_parameter_names
    and not any(
        parameter.kind == inspect.Parameter.VAR_KEYWORD
        for parameter in forward_signature.parameters.values()
    )
]

if missing_forward_keys:
    raise RuntimeError(
        "The model forward API is missing required training inputs: "
        f"{missing_forward_keys}"
    )


model.forward signature: (*args: 'Any', **kwargs: 'Any')
Trainer batch keys: ['attention_mask', 'image_flags', 'input_ids', 'labels', 'pixel_values']
Confirmed: num_patches_list is inference/collator metadata only and will not be passed to model.forward().


### Trainer forward API fix

`num_patches_list` belongs to the inference route:

```python
model.chat(
    ...,
    num_patches_list=[2, 2, 2],
)
```

The training route calls:

```python
model.forward(
    input_ids=...,
    attention_mask=...,
    labels=...,
    pixel_values=...,
    image_flags=...,
)
```

`InternVLChatModel.forward()` in this environment does not declare
`num_patches_list`. The previous collator returned that key, so Hugging Face
Trainer forwarded it directly and Python raised:

```text
TypeError: InternVLChatModel.forward() got an unexpected keyword argument
'num_patches_list'
```

The fixed collator still computes and validates patch groups internally, but it
returns only keys accepted by the training forward path.


### Transformers Trainer API compatibility

In [23]:
import inspect

def build_transformers_trainer(
    *,
    model,
    args,
    train_dataset,
    data_collator,
    tokenizer,
):
    """Instantiate Trainer across old and new Transformers APIs.

    Older versions accept:
        tokenizer=...

    Newer versions accept:
        processing_class=...

    This helper inspects the active Trainer signature and passes only
    supported arguments.
    """
    trainer_signature = inspect.signature(Trainer.__init__)
    trainer_parameters = trainer_signature.parameters

    trainer_kwargs = {
        "model": model,
        "args": args,
        "train_dataset": train_dataset,
        "data_collator": data_collator,
    }

    if "processing_class" in trainer_parameters:
        trainer_kwargs["processing_class"] = tokenizer
        tokenizer_parameter_used = "processing_class"
    elif "tokenizer" in trainer_parameters:
        trainer_kwargs["tokenizer"] = tokenizer
        tokenizer_parameter_used = "tokenizer"
    else:
        tokenizer_parameter_used = None

    print("Trainer.__init__ signature:", trainer_signature)
    print("Tokenizer/processor argument used:", tokenizer_parameter_used)
    print("Trainer kwargs:", sorted(trainer_kwargs.keys()))

    return Trainer(**trainer_kwargs)


In [24]:
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=FP16,
    bf16=BF16,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

assert training_args.fp16
assert not training_args.bf16
assert not training_args.gradient_checkpointing

trainer = build_transformers_trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

if trainer.optimizer is not None:
    raise RuntimeError("Optimizer should not exist before trainer.train().")

cleanup_all_memory(reset_cuda_peak_stats=True)
print_system_memory("immediately before trainer.train()")
print_cuda_memory("immediately before trainer.train()")
torch.cuda.reset_peak_memory_stats()
train_started_at = time.time()

train_result = trainer.train()

train_runtime_seconds = time.time() - train_started_at
peak_vram_gb = torch.cuda.max_memory_allocated() / 1024**3

trainer.save_model(str(CHECKPOINT_DIR / "final_lora_adapter"))

RUN_STATUS["training_completed"] = True

print("Train runtime seconds:", round(train_runtime_seconds, 2))
print("Peak allocated VRAM GB:", round(peak_vram_gb, 2))


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer.__init__ signature: (self, model: transformers.modeling_utils.PreTrainedModel | torch.nn.modules.module.Module | None = None, args: transformers.training_args.TrainingArguments | None = None, data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None, train_dataset: 'Dataset | IterableDataset | datasets.Dataset | None' = None, eval_dataset: 'Dataset | dict[str, Dataset] | datasets.Dataset | None' = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.image_processing_utils.BaseImageProcessor | transformers.feature_extraction_utils.FeatureExtractionMixin | transformers.processing_utils.ProcessorMixin | None = None, model_init: collections.abc.Callable[..., transformers.modeling_utils.PreTrainedModel] | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callbacks: list[

[SYSTEM RAM] after RAM cleanup: {'total_gib': 94.0, 'available_gib': 86.94076919555664, 'used_gib': 7.059230804443359, 'available_ratio': 0.9249017999527303, 'percent_used': 7.5, 'process_rss_gib': 1.7062416076660156, 'process_vms_gib': 22.57735824584961}


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'pad_token_id': 151643}.


[CUDA MEMORY] after cleanup: {'cuda_available': True, 'device': 'Tesla V100-SXM2-32GB', 'free_gib': 29.18212890625, 'total_gib': 31.7325439453125, 'allocated_gib': 1.7805814743041992, 'reserved_gib': 1.873046875, 'max_allocated_gib': 1.7805814743041992, 'free_ratio': 0.9196277788677184}
[SYSTEM RAM] immediately before trainer.train(): {'total_gib': 94.0, 'available_gib': 86.94095230102539, 'used_gib': 7.059047698974609, 'available_ratio': 0.9249037478832488, 'percent_used': 7.5, 'process_rss_gib': 1.7062416076660156, 'process_vms_gib': 22.51485824584961}
[CUDA MEMORY] immediately before trainer.train(): {'cuda_available': True, 'device': 'Tesla V100-SXM2-32GB', 'free_gib': 29.18212890625, 'total_gib': 31.7325439453125, 'allocated_gib': 1.7805814743041992, 'reserved_gib': 1.873046875, 'max_allocated_gib': 1.7805814743041992, 'free_ratio': 0.9196277788677184}


...

Local multi-frame tile cap: {'original_patch_counts': [3, 3, 3], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}, {'frame_index': 1, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}, {'frame_index': 2, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}


Local multi-frame tile cap: {'original_patch_counts': [3, 3, 3], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}, {'frame_index': 1, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}, {'frame_index': 2, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}


Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}


Local multi-frame tile cap: {'original_patch_counts': [3, 3, 3], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}, {'frame_index': 1, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}, {'frame_index': 2, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}


Train runtime seconds: 424.86
Peak allocated VRAM GB: 11.82


## 12. Verify LoRA updates

In [25]:
lora_delta_rows = []

for name, parameter in model.named_parameters():
    if name not in initial_lora_state:
        continue

    delta_norm = float(
        (
            parameter.detach().cpu().float()
            - initial_lora_state[name].float()
        ).norm().item()
    )

    lora_delta_rows.append({
        "parameter_name": name,
        "delta_norm": delta_norm,
        "changed": delta_norm > 0.0,
    })

lora_delta_df = pd.DataFrame(lora_delta_rows)
changed_count = int(lora_delta_df["changed"].sum())

lora_delta_df.to_csv(
    LOG_DIR / f"{RUN_ID}_lora_deltas.csv",
    index=False,
)

if changed_count <= 0:
    raise RuntimeError("LoRA weights did not change.")

RUN_STATUS["lora_updated"] = True

print("Changed LoRA tensors:", changed_count, "/", len(lora_delta_df))


Changed LoRA tensors: 192 / 192


## 13. Fixed validation evaluation

In [26]:
def normalize_choice(value):
    text = "" if value is None else str(value).strip().upper()
    match = re.search(
        r"(?:^|ĐÁP\s*ÁN\s*[:\-]?\s*|\b)([A-D])(?:\b|[\.\):\-])",
        text,
    )
    return match.group(1) if match else None


evaluation_records = (
    val_records[:DEBUG_VAL_SAMPLES]
    if DEBUG_MODE
    else val_records
)

model.eval()
validation_rows = []
validation_started_at = time.time()

for row in tqdm(evaluation_records, desc=f"Validation {RUN_ID}"):
    pixels, patch_counts, timestamps = preprocess_multiframe(
        row,
        target_model=model,
    )

    multiframe_question = build_multiframe_question(
        row["question"],
        patch_counts,
    )
    validate_multiframe_prompt(
        multiframe_question,
        patch_counts,
    )

    result = strict_multimodal_generate(
        model=model,
        tokenizer=tokenizer,
        question=multiframe_question,
        pixel_values=pixels,
        num_patches_list=patch_counts,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    prediction = str(result["response"]).strip()
    reference = str(row["answer"]).strip()

    normalized_prediction = normalize_choice(prediction)
    normalized_reference = normalize_choice(reference)

    validation_rows.append({
        "id": row["id"],
        "video_id": row["video_id"],
        "video_path": row["video_path"],
        "question": row["question"],
        "image_placeholder_count": multiframe_question.count("<image>"),
        "ground_truth": reference,
        "raw_prediction": prediction,
        "normalized_ground_truth": normalized_reference,
        "normalized_prediction": normalized_prediction,
        "normalized_correct": (
            normalized_prediction is not None
            and normalized_reference is not None
            and normalized_prediction == normalized_reference
        ),
        "raw_exact_match": prediction == reference,
        "is_empty": prediction == "",
        "parse_failed": normalized_prediction is None,
        "patch_counts": json.dumps(patch_counts),
        "selected_timestamps": json.dumps(timestamps),
    })

validation_runtime_seconds = time.time() - validation_started_at
validation_df = pd.DataFrame(validation_rows)

metrics = {
    "run_id": RUN_ID,
    "debug_mode": DEBUG_MODE,
    "sample_count": int(len(validation_df)),
    "normalized_choice_accuracy": float(
        validation_df["normalized_correct"].mean()
    ),
    "raw_exact_match_accuracy": float(
        validation_df["raw_exact_match"].mean()
    ),
    "empty_output_rate": float(validation_df["is_empty"].mean()),
    "parse_failure_rate": float(validation_df["parse_failed"].mean()),
    "train_runtime_seconds": float(train_runtime_seconds),
    "validation_runtime_seconds": float(validation_runtime_seconds),
    "peak_vram_gb": float(peak_vram_gb),
    "changed_lora_tensors": changed_count,
}

validation_df.to_csv(
    PREDICTION_DIR / f"{RUN_ID}_validation.csv",
    index=False,
)
(METRICS_DIR / f"{RUN_ID}.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

if metrics["empty_output_rate"] != 0.0:
    raise RuntimeError("Phase 2 MVP produced empty validation outputs.")

RUN_STATUS["validation_completed"] = True

print(json.dumps(metrics, ensure_ascii=False, indent=2))


Validation phase2_f3_uniform_v1:   0%|          | 0/64 [00:00<?, ?it/s]

Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}
Validated <IMG_CONTEXT> token ID: 151667


Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}
Validated <IMG_CONTEXT> token ID: 151667


Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}
Validated <IMG_CONTEXT> token ID: 151667


Local multi-frame tile cap: {'original_patch_counts': [5, 5, 5], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 1, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}, {'frame_index': 2, 'original_count': 5, 'selected_local_indices': [0, 4], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}
Validated <IMG_CONTEXT> token ID: 151667


...

Local multi-frame tile cap: {'original_patch_counts': [3, 3, 3], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}, {'frame_index': 1, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}, {'frame_index': 2, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}
Validated <IMG_CONTEXT> token ID: 151667


Local multi-frame tile cap: {'original_patch_counts': [3, 3, 3], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}, {'frame_index': 1, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}, {'frame_index': 2, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}
Validated <IMG_CONTEXT> token ID: 151667


Local multi-frame tile cap: {'original_patch_counts': [3, 3, 3], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}, {'frame_index': 1, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}, {'frame_index': 2, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}
Validated <IMG_CONTEXT> token ID: 151667


Local multi-frame tile cap: {'original_patch_counts': [3, 3, 3], 'capped_patch_counts': [2, 2, 2], 'selection_log': [{'frame_index': 0, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}, {'frame_index': 1, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}, {'frame_index': 2, 'original_count': 3, 'selected_local_indices': [0, 2], 'selected_count': 2}], 'total_tiles': 6, 'total_budget': 6}
Validated <IMG_CONTEXT> token ID: 151667


{
  "run_id": "phase2_f3_uniform_v1",
  "debug_mode": true,
  "sample_count": 64,
  "normalized_choice_accuracy": 0.125,
  "raw_exact_match_accuracy": 0.046875,
  "empty_output_rate": 0.0,
  "parse_failure_rate": 0.421875,
  "train_runtime_seconds": 424.8593463897705,
  "validation_runtime_seconds": 84.52343392372131,
  "peak_vram_gb": 11.822735786437988,
  "changed_lora_tensors": 192
}


## 14. Compare against Phase 1 baseline

In [27]:
phase1_metrics_path = (
    PHASE1_ARTIFACT_DIR / "validation_metrics_phase1.json"
)

if phase1_metrics_path.is_file():
    phase1_metrics = json.loads(
        phase1_metrics_path.read_text(encoding="utf-8")
    )

    comparison = pd.DataFrame([
        {
            "run_id": "phase1_f1_baseline",
            "num_frames": 1,
            "sample_count": phase1_metrics.get("sample_count"),
            "normalized_choice_accuracy": phase1_metrics.get(
                "normalized_choice_accuracy"
            ),
            "empty_output_rate": phase1_metrics.get("empty_output_rate"),
        },
        {
            "run_id": RUN_ID,
            "num_frames": NUM_FRAMES,
            "sample_count": metrics["sample_count"],
            "normalized_choice_accuracy": metrics[
                "normalized_choice_accuracy"
            ],
            "empty_output_rate": metrics["empty_output_rate"],
        },
    ])

    # A strict accuracy comparison is valid only when both runs use the same
    # validation sample IDs. In DEBUG_MODE this table is informational only.
    comparison["comparison_note"] = (
        "Informational only in DEBUG_MODE; run full validation for final claim."
        if DEBUG_MODE
        else "Full Phase 1 split comparison."
    )

    comparison.to_csv(
        METRICS_DIR / f"{RUN_ID}_vs_phase1.csv",
        index=False,
    )
    display(comparison)
else:
    print("Phase 1 metrics file not found:", phase1_metrics_path)


,run_id,num_frames,sample_count,normalized_choice_accuracy,empty_output_rate,comparison_note
0,phase1_f1_baseline,1,319,0.338558,0.0,Informational only in DEBUG_MODE; run full val...
1,phase2_f3_uniform_v1,3,64,0.125000,0.0,Informational only in DEBUG_MODE; run full val...


## 15. MVP final status

In [28]:
required_checks = [
    "phase1_split_loaded",
    "model_loaded",
    "multiframe_smoke_passed",
    "dataset_probe_passed",
    "collator_probe_passed",
    "lora_ready",
    "training_completed",
    "lora_updated",
    "validation_completed",
]

failed_checks = [
    key for key in required_checks
    if not RUN_STATUS.get(key, False)
]

mvp_report = {
    "run_id": RUN_ID,
    "status": "PASSED" if not failed_checks else "FAILED",
    "checks": RUN_STATUS,
    "failed_checks": failed_checks,
    "config": run_config,
    "metrics": metrics,
    "artifacts": {
        "config": str(CONFIG_DIR / f"{RUN_ID}.json"),
        "checkpoint": str(CHECKPOINT_DIR / "final_lora_adapter"),
        "metrics": str(METRICS_DIR / f"{RUN_ID}.json"),
        "predictions": str(
            PREDICTION_DIR / f"{RUN_ID}_validation.csv"
        ),
    },
}

report_path = RUN_DIR / "PHASE2_MVP_STATUS.json"
report_path.write_text(
    json.dumps(mvp_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(mvp_report, ensure_ascii=False, indent=2))

if failed_checks:
    raise RuntimeError(f"Phase 2 MVP failed checks: {failed_checks}")

print("PHASE 2 MVP STATUS: PASSED")
print("Report:", report_path)


{
  "run_id": "phase2_f3_uniform_v1",
  "status": "PASSED",
  "checks": {
    "phase1_split_loaded": true,
    "model_loaded": true,
    "multiframe_smoke_passed": true,
    "dataset_probe_passed": true,
    "collator_probe_passed": true,
    "lora_ready": true,
    "training_completed": true,
    "lora_updated": true,
    "validation_completed": true
  },
  "failed_checks": [],
  "config": {
    "run_id": "phase2_f3_uniform_v1",
    "seed": 42,
    "debug_mode": true,
    "debug_train_samples": 256,
    "debug_val_samples": 64,
    "num_frames": 3,
    "sampling_strategy": "uniform",
    "max_tiles_per_frame": 2,
    "max_total_tiles": 6,
    "input_size": 448,
    "max_length": 2048,
    "max_new_tokens": 8,
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "learning_rate": 0.0001,
    "epochs": 1
  },
  "metrics": {
    "run_id": "phase2_f3_uniform_v1",
    "debug_mode": true,
    "sample_count": 64,
    "normalized_choice_accuracy": 0.125,
    "raw_exact_matc

## Stable Phase 2 artifact contract

In [29]:
# =========================
# PHASE 2 ARTIFACT CONTRACT
# =========================

from hashlib import sha256

RUN_OUTPUT_DIR = Path("./outputs/roadbuddy_phase2") / RUN_ID
CONFIG_DIR = RUN_OUTPUT_DIR / "configs"
METRICS_DIR = RUN_OUTPUT_DIR / "metrics"
PREDICTIONS_DIR = RUN_OUTPUT_DIR / "predictions"
CHECKPOINT_DIR = RUN_OUTPUT_DIR / "checkpoints"

for directory in [
    RUN_OUTPUT_DIR,
    CONFIG_DIR,
    METRICS_DIR,
    PREDICTIONS_DIR,
    CHECKPOINT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

def stable_ids_hash(values):
    payload = "\n".join(map(str, values)).encode("utf-8")
    return sha256(payload).hexdigest()

validation_ids = []
for record in val_records:
    validation_ids.append(
        record.get("id")
        or record.get("sample_id")
        or record.get("question_id")
        or record.get("video_id")
        or str(len(validation_ids))
    )

validation_ids_path = (
    RUN_OUTPUT_DIR / "validation_sample_ids.json"
)
validation_ids_path.write_text(
    json.dumps(validation_ids, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

config_payload = {
    "run_id": RUN_ID,
    "stage": STAGE,
    "target_set": TARGET_SET,
    "debug_mode": DEBUG_MODE,
    "debug_train_samples": DEBUG_TRAIN_SAMPLES,
    "debug_val_samples": DEBUG_VAL_SAMPLES,
    "num_frames": NUM_FRAMES,
    "sampling_strategy": SAMPLING_STRATEGY,
    "max_tiles_per_frame": MAX_TILES_PER_FRAME,
    "max_total_tiles": MAX_TOTAL_TILES,
    "input_size": INPUT_SIZE,
    "max_length": MAX_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "learning_rate": LEARNING_RATE,
    "epochs": NUM_TRAIN_EPOCHS,
    "validation_ids_hash": stable_ids_hash(validation_ids),
    "runtime": RUNTIME_METADATA,
}

config_path = CONFIG_DIR / f"{RUN_ID}.json"
config_path.write_text(
    json.dumps(config_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Artifact config:", config_path)
print("Validation IDs:", validation_ids_path)


Artifact config: outputs/roadbuddy_phase2/phase2_f3_uniform_v1/configs/phase2_f3_uniform_v1.json
Validation IDs: outputs/roadbuddy_phase2/phase2_f3_uniform_v1/validation_sample_ids.json


## Final GPU memory cleanup

### Robust answer parser

In [30]:
import re

def parse_choice_robust(text):
    """Parse A/B/C/D from concise or lightly formatted model outputs."""
    if text is None:
        return None

    normalized = str(text).strip().upper()

    patterns = [
        r"^\s*[\(\[]?([ABCD])[\)\]\.\:\-]?\s*$",
        r"(?:ĐÁP\s*ÁN|ANSWER|LỰA\s*CHỌN|CHOICE)\s*[:\-]?\s*[\(\[]?([ABCD])",
        r"\b([ABCD])\b",
    ]

    for pattern in patterns:
        match = re.search(pattern, normalized)
        if match:
            return match.group(1)

    return None


In [31]:
print_system_memory("before final cleanup")
print_cuda_memory("before final cleanup")

delete_globals_and_cleanup(
    "trainer",
    "train_result",
    "model",
    "initial_lora_state",
    "lora_delta_df",
    "validation_df",
    "validation_rows",
    "train_dataset",
    "train_dataset_full",
    "val_dataset",
    "val_dataset_full",
    "train_records",
    "val_records",
    "all_records",
    "collator_probe",
    "smoke_pixels",
    "smoke_result",
)

cleanup_all_memory(reset_cuda_peak_stats=True)

print_system_memory("final state")
print_cuda_memory("final state")
print("Notebook RAM and VRAM cleanup completed.")


[SYSTEM RAM] before final cleanup: {'total_gib': 94.0, 'available_gib': 87.09179306030273, 'used_gib': 6.908206939697266, 'available_ratio': 0.9265084368117312, 'percent_used': 7.3, 'process_rss_gib': 1.9837455749511719, 'process_vms_gib': 40.69380187988281}
[CUDA MEMORY] before final cleanup: {'cuda_available': True, 'device': 'Tesla V100-SXM2-32GB', 'free_gib': 12.35205078125, 'total_gib': 31.7325439453125, 'allocated_gib': 1.8113598823547363, 'reserved_gib': 18.693359375, 'max_allocated_gib': 11.822735786437988, 'free_ratio': 0.3892549807080511}
Deleted: trainer
Deleted: train_result
Deleted: model
Deleted: initial_lora_state
Deleted: lora_delta_df
Deleted: validation_df
Deleted: validation_rows
Deleted: train_dataset
Deleted: train_dataset_full
Deleted: val_dataset
Deleted: val_dataset_full
Deleted: train_records
Deleted: val_records
Deleted: all_records
Deleted: collator_probe
Deleted: smoke_pixels
Deleted: smoke_result


[CUDA MEMORY] after cleanup: {'cuda_available': True, 'device': 'Tesla V100-SXM2-32GB', 'free_gib': 29.10986328125, 'total_gib': 31.7325439453125, 'allocated_gib': 1.7885160446166992, 'reserved_gib': 1.935546875, 'max_allocated_gib': 1.7885160446166992, 'free_ratio': 0.9173504441187446}


[SYSTEM RAM] after RAM cleanup: {'total_gib': 94.0, 'available_gib': 87.15369033813477, 'used_gib': 6.846309661865234, 'available_ratio': 0.9271669184907954, 'percent_used': 7.3, 'process_rss_gib': 1.9131317138671875, 'process_vms_gib': 24.662429809570312}


[CUDA MEMORY] after cleanup: {'cuda_available': True, 'device': 'Tesla V100-SXM2-32GB', 'free_gib': 29.10986328125, 'total_gib': 31.7325439453125, 'allocated_gib': 1.7885160446166992, 'reserved_gib': 1.935546875, 'max_allocated_gib': 1.7885160446166992, 'free_ratio': 0.9173504441187446}
[SYSTEM RAM] final state: {'total_gib': 94.0, 'available_gib': 87.1536636352539, 'used_gib': 6.846336364746094, 'available_ratio': 0.9271666344175947, 'percent_used': 7.3, 'process_rss_gib': 1.9131317138671875, 'process_vms_gib': 24.662429809570312}
[CUDA MEMORY] final state: {'cuda_available': True, 'device': 'Tesla V100-SXM2-32GB', 'free_gib': 29.10986328125, 'total_gib': 31.7325439453125, 'allocated_gib': 1.7885160446166992, 'reserved_gib': 1.935546875, 'max_allocated_gib': 1.7885160446166992, 'free_ratio': 0.9173504441187446}
Notebook RAM and VRAM cleanup completed.
